# ECON 3916 -- Checkpoint Summary
## Ian Solberg | April 19, 2026

---

## Proposal

**Prediction question (1 sentence):**
Can observable macroeconomic indicators predict the year-over-year growth rate of U.S. average weekly earnings for private-sector employees?

**Prediction vs. causation distinction:**
This is a prediction problem, not a causal one. We are asking "can we forecast Y from X?" rather than "does X cause Y?" The models identify which features are *predictively associated* with wage growth, but a high-importance feature does not mean changes in that feature *cause* changes in wages. For example, Labor Force Participation may be the most important predictor in Gradient Boosting, but that reflects correlation and regime co-movement, not a causal mechanism. Establishing causation would require an experimental or quasi-experimental design (e.g., instrumental variables, difference-in-differences), which is outside the scope of this project.

**Dataset:**
- **Source:** Federal Reserve Economic Data (FRED) API via custom `FRED_Loader` submodule
- **URL:** [https://fred.stlouisfed.org/](https://fred.stlouisfed.org/)
- **N:** 998 weekly observations (research subset, March 2007 -- April 2026)
- **Features:** 32 in the research subset (selected from 229 raw series + 75 derived scoring columns)
- **Target variable:** `Avg_Weekly_Earnings_YoY` -- year-over-year percent change in average weekly earnings for all private-sector employees (BLS series CES0500000011)
- **Access date:** April 15, 2026

**Stakeholder:**
This analysis would help *monetary policy analysts at the Federal Reserve and labor economists at the Bureau of Labor Statistics* decide *whether current macroeconomic conditions signal upcoming wage acceleration or deceleration, informing interest rate decisions, inflation forecasts, and labor market monitoring*.

---

## EDA

**Data loaded and described:**
- Shape: (998, 34) -- 998 rows, 34 columns (33 features + 1 target)
- Dtypes: 31 float64, 2 int64, 1 string (date)
- Date range: 2007-03-02 to 2026-04-11

**Missing data assessment:**
- Missing data in the research subset: **0%** across all 34 columns
- Strategy: The upstream FRED_Loader pipeline handles missingness via last-observation-carried-forward (LOCF) at the weekly resampling stage. Monthly and quarterly series are forward-filled to weekly frequency. A datetime trim removes stale forward-filled rows beyond the last real FRED release.
- Classification under Ch 1 framework: **MAR (Missing At Random)** -- missingness is conditional on the publication schedule of each FRED series (monthly series have no values between release dates), not on the values themselves. This is not MCAR because the pattern is systematic (always between releases), and not MNAR because the reason for missingness is the publication calendar, not the unobserved value.

**Visualizations (3+):**
1. **Missing data heatmap** -- confirms 0% missingness across all columns in the research subset
2. **Target distribution (histogram + KDE)** -- shows Avg_Weekly_Earnings_YoY centered around 2-4% with a COVID-era spike to ~7.5%, approximately unimodal after the YoY transformation
3. **Correlation heatmap** -- reveals feature clustering (labor market block, inflation block, financial conditions block) and identifies the 12 highest-correlation features for the final model
4. **Time series plot of target** -- shows the full 2007-2026 trajectory including the post-GFC recovery, pre-COVID stability, COVID spike, and post-pandemic deceleration

**Data quality summary:**
- No missing values in the research subset
- No duplicate rows (each row is a unique Friday-aligned week)
- Target variable is stationary after YoY transformation (confirmed visually and by the absence of trend in the autocorrelation structure)
- Three level variables (Retail_Sales, JOLTS_Quits, Part_Time_Economic_Reasons) were flagged as non-stationary and replaced with scored stationary alternatives before final modeling
- Outlier assessment: COVID-era spike (2020-2021) represents a genuine structural event, not a data error; retained in training data

---

## Preliminary Model

**Train/test split:**
- Temporal split at 80/20: ~788 train (through July 2022), ~197 test (July 2022 -- April 2026)
- `random_state=42` used for all random operations
- Temporal split chosen over random split to prevent data leakage from future observations

**Model fitted:**
- Baseline: OLS Linear Regression (scikit-learn `LinearRegression`)
- Also fitted: Ridge Regression (alpha=1.0), Gradient Boosting Regressor (n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8)

**Metrics reported (regression):**

| Model | RMSE | MAE | R-squared |
|-------|------|-----|-----------|
| OLS | 0.3977 | 0.3169 | 0.011 |
| Ridge (alpha=1.0) | 0.3965 | 0.3162 | 0.017 |
| Gradient Boosting | 0.4525 | 0.3763 | -0.280 |

All metrics computed on the temporal test set (July 2022 -- April 2026).

---

## GitHub

- **Repository:** [https://github.com/iasolb/ML_Prediction_Project](https://github.com/iasolb/ML_Prediction_Project)
- **Status:** Public (or private with instructor access)
